<a href="https://colab.research.google.com/github/MohammadAhmadSiddiqui/28-March_Capstone_Project/blob/main/Scenario_2_AI_Career_Advisor_%26_Learning_Path_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔷 Scenario 2: AI Career Advisor & Learning Path Generator

🎯 Objective:

Build a multi-step intelligent advisor system using agent workflows.

💼 Problem Statement

Students need guidance on career paths based on their skills.

Your system should:

Analyze student profile
Suggest career roles
Generate learning roadmap

👥 Agents to Build

Profile Analyzer Agent
Understands student background
Career Recommendation Agent
Suggests suitable roles
Skill Gap Agent
Identifies missing skills
Learning Path Agent
Creates step-by-step roadmap

⚙️ Tasks to Implement

Sequential workflow (LangGraph preferred)
State passing between agents
Conditional logic (beginner vs intermediate)

💡 Sample Input

“I know Python basics and statistics, and I want to become a Data Scientist”

🧪 Expected Output

Suggested roles (e.g., Data Scientist, Analyst)
Skill gaps (ML, SQL, etc.)
Learning roadmap (step-by-step)

🧰 Tools (Choose Any)

LangGraph (recommended)

CrewAI

AutoGen

Python logic-based workflow

Microsoft Copilot Studio

📦 Deliverables

Workflow implementation

Output for at least 2 student profiles

In [ ]:
!pip install autogen
!pip install python-dotenv

In [5]:
import os
import autogen
from dotenv import load_dotenv
from google.colab import userdata
api_key=userdata.get('GROQ_API_KEY')

# 1. Configuration (Pointing to Groq)
config_list = [
    {
        "model": "llama-3.3-70b-versatile",
        "api_key": api_key,
        "base_url": "https://api.groq.com/openai/v1",
    }
]

llm_config = {"config_list": config_list, "temperature": 0.3}

# AGENT DEFINITIONS (LLM-Powered Personas)

# The Interface (Collects user input)
user_proxy = autogen.UserProxyAgent(
    name="Admin",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=0,
    code_execution_config=False
)

analyzer = autogen.AssistantAgent(
    name="Profile_Analyzer",
    llm_config=llm_config,
    system_message="You are a Profile Analyzer. Identify the user's current technical stack and their career ambition. Summarize these as a 'Student Profile'."
)

recommender = autogen.AssistantAgent(
    name="Career_Recommender",
    llm_config=llm_config,
    system_message="You are a Career Strategist. Based on the Student Profile, suggest the top 2 trending job roles they should target."
)

gap_agent = autogen.AssistantAgent(
    name="Skill_Gap_Agent",
    llm_config=llm_config,
    system_message="You are a Technical Recruiter. Compare the user's current skills to the requirements of the suggested roles. Identify exactly what is missing (Skill Gaps)."
)

roadmap_agent = autogen.AssistantAgent(
    name="Learning_Path_Agent",
    llm_config=llm_config,
    system_message="You are a Learning Architect. Create a month-by-month roadmap to bridge the Skill Gaps. If they are a 'Beginner', include fundamentals. If 'Intermediate', focus on advanced projects."
)

# ============================================
# THE DYNAMIC WORKFLOW FUNCTION
# ============================================

def run_dynamic_advisor(user_query):
    print(f"\n [System]: Initializing Agents for: {user_query}")

    # Sequence: Analyzer -> Recommender -> Gap Agent -> Roadmap Agent

    # Step 1: Analyze Profile
    user_proxy.initiate_chat(analyzer, message=user_query, clear_history=True)
    profile_data = user_proxy.last_message(analyzer)["content"]

    # Step 2: Recommend Roles
    user_proxy.initiate_chat(recommender, message=profile_data, clear_history=True)
    roles_data = user_proxy.last_message(recommender)["content"]

    # Step 3: Identify Gaps
    combined_context = f"Profile: {profile_data}\nTarget Roles: {roles_data}"
    user_proxy.initiate_chat(gap_agent, message=combined_context, clear_history=True)
    gap_data = user_proxy.last_message(gap_agent)["content"]

    # Step 4: Generate Roadmap
    user_proxy.initiate_chat(roadmap_agent, message=gap_data, clear_history=True)
    final_roadmap = user_proxy.last_message(roadmap_agent)["content"]

    # Final Structured Output Agent
    return f"""
==================================================
 AI CAREER ADVISOR REPORT
==================================================
{profile_data}

--------------------------------------------------
 SUGGESTED ROLES
{roles_data}

--------------------------------------------------
 SKILL GAPS IDENTIFIED
{gap_data}

--------------------------------------------------
 LEARNING ROADMAP
{final_roadmap}
==================================================
"""

# ============================================
# DYNAMIC RUN LOOP
# ============================================
print(" AI Career Advisor System Started (type 'exit')\n")

while True:
    student_input = input("Enter your skills and career goal: ").strip()

    if student_input.lower() in ["exit", "quit"]:
        print("Best of luck with your career!")
        break

    if not student_input:
        continue

    try:
        report = run_dynamic_advisor(student_input)
        print(report)
    except Exception as e:
        print(f" Error: {e}")

 AI Career Advisor System Started (type 'exit')

Enter your skills and career goal: I know Ai-Ml

 [System]: Initializing Agents for: I know Ai-Ml
Admin (to Profile_Analyzer):

I know Ai-Ml

--------------------------------------------------------------------------------
[autogen.oai.client: 03-28 09:16:44] {744} WARNING - Model llama-3.3-70b-versatile is not found. The cost will be 0. In your config_list, add field {"price" : [prompt_price_per_1k, completion_token_price_per_1k]} for customized pricing.


Profile_Analyzer (to Admin):

**Student Profile**

Based on the information provided, here is a summary of your profile:

* **Current Technical Stack:** AI-ML (Artificial Intelligence and Machine Learning)
* **Career Ambition:** Not explicitly stated, but potential career ambitions related to AI-ML may include:
	+ Data Scientist
	+ Machine Learning Engineer
	+ AI Researcher
	+ Business Intelligence Developer
	+ Natural Language Processing Specialist

To further refine your profile, could you please provide more information about your career goals and aspirations? What specific areas within AI-ML interest you the most? Are you looking to work in a particular industry or sector?

--------------------------------------------------------------------------------

>>>>>>>> TERMINATING RUN (47a9bb05-dc11-414f-bf44-c054b1fe56b5): Maximum number of consecutive auto-replies reached
Admin (to Career_Recommender):

**Student Profile**

Based on the information provided, here is a summary of your p

Career_Recommender (to Admin):

Based on the provided Student Profile, I'll suggest the top 2 trending job roles that align with their current technical stack in AI-ML. Since the student's career ambitions are not explicitly stated, I'll consider the most in-demand and versatile roles in the industry.

**Top 2 Trending Job Roles:**

1. **Machine Learning Engineer**: This role involves designing, developing, and deploying machine learning models to solve complex problems. With a strong foundation in AI-ML, the student can leverage their skills to work on projects that require model development, data preprocessing, and model deployment. Machine Learning Engineers are in high demand across various industries, including healthcare, finance, and technology.
2. **Data Scientist (AI-ML focus)**: As a Data Scientist with a specialization in AI-ML, the student can work on extracting insights from large datasets, developing predictive models, and creating data visualizations. This role requires 

Skill_Gap_Agent (to Admin):

**Comparison of Current Skills to Target Roles:**

Based on the provided Student Profile, we can compare the current technical stack in AI-ML to the requirements of the suggested roles:

1. **Machine Learning Engineer:**
	* Current skills: AI-ML (Artificial Intelligence and Machine Learning)
	* Required skills:
		+ Programming languages (e.g., Python, R, Julia)
		+ Machine learning frameworks (e.g., TensorFlow, PyTorch, Scikit-learn)
		+ Data preprocessing and feature engineering
		+ Model deployment and maintenance
	* Skill gaps:
		+ Programming languages (specifically, proficiency in Python or other relevant languages)
		+ Experience with machine learning frameworks
		+ Knowledge of data preprocessing and feature engineering techniques
		+ Familiarity with model deployment and maintenance tools (e.g., Docker, Kubernetes)
2. **Data Scientist (AI-ML focus):**
	* Current skills: AI-ML (Artificial Intelligence and Machine Learning)
	* Required skills:
		+ Pro

Learning_Path_Agent (to Admin):

**Month-by-Month Roadmap to Bridge Skill Gaps**

Based on the identified skill gaps, I've created a month-by-month roadmap to help the student acquire the necessary skills for the Machine Learning Engineer and Data Scientist (AI-ML focus) roles. This roadmap assumes the student is a beginner in programming languages and machine learning frameworks.

**Month 1: Programming Fundamentals (Weeks 1-4)**

* Week 1: Introduction to Python programming (Codecademy, Coursera)
	+ Complete basic Python courses and exercises
	+ Focus on data types, variables, control structures, functions, and modules
* Week 2: Practice Python programming (LeetCode, HackerRank)
	+ Solve problems and exercises to improve coding skills
	+ Focus on data structures (lists, dictionaries, sets) and file input/output
* Week 3: Introduction to R programming (Codecademy, Coursera)
	+ Complete basic R courses and exercises
	+ Focus on data types, variables, control structures, functions, and 

Profile_Analyzer (to Admin):

**Student Profile**

Based on your interest in learning Web Development, I've created a profile to summarize your current technical stack and career ambition.

**Current Technical Stack:**
- Programming languages: None specified ( beginner level)
- Development frameworks: None specified
- Databases: None specified
- Operating System: Not specified
- Familiarity with Web Development: Beginner

**Career Ambition:**
- Desired career path: Web Development
- Job role: Junior Web Developer or Front-end/Back-end Developer
- Industry: Technology and software development
- Career goals: Gain skills and knowledge in Web Development to secure a job as a Web Developer and continue to grow in the field.

As you progress in your learning journey, this profile can be updated to reflect your growing technical stack and career ambitions. What specific areas of Web Development are you interested in (Front-end, Back-end, Full-stack)?

----------------------------------------

Career_Recommender (to Admin):

Based on your Student Profile, I'd like to suggest the top 2 trending job roles that you should target as a beginner in Web Development:

1. **Front-end Developer**: As a beginner, focusing on Front-end Development can be a great starting point. Front-end Developers are responsible for creating the user interface and user experience of a website or web application using programming languages like HTML, CSS, and JavaScript. With the growing demand for responsive and interactive web applications, Front-end Developers are in high demand. You can start by learning the basics of HTML, CSS, and JavaScript, and then move on to popular front-end frameworks like React, Angular, or Vue.js.
2. **Full-stack Developer**: As you gain more experience and skills in Web Development, targeting a Full-stack Developer role can be a great career goal. Full-stack Developers work on both the front-end and back-end of a website or web application, using a range of programming l

Skill_Gap_Agent (to Admin):

**Comparison of Current Skills to Target Roles:**

Based on your Student Profile, I've compared your current technical stack to the requirements of the suggested roles. Here's a summary of the skill gaps:

**Front-end Developer:**

* Required skills:
	+ Programming languages: HTML, CSS, JavaScript
	+ Development frameworks: React, Angular, or Vue.js
* Your current skills:
	+ Programming languages: None specified (beginner level)
	+ Development frameworks: None specified
* Skill gaps:
	+ HTML
	+ CSS
	+ JavaScript
	+ Front-end frameworks (React, Angular, or Vue.js)

**Full-stack Developer:**

* Required skills:
	+ Programming languages: HTML, CSS, JavaScript (front-end), Python, Ruby, or PHP (back-end)
	+ Development frameworks: Django, Ruby on Rails, or Laravel (back-end), React, Angular, or Vue.js (front-end)
	+ Databases: Various databases such as MySQL, MongoDB, or PostgreSQL
* Your current skills:
	+ Programming languages: None specified (beginner level)

Learning_Path_Agent (to Admin):

**Month-by-Month Roadmap to Bridge Skill Gaps**

As a beginner, it's essential to start with the fundamentals and gradually build upon them. Here's a month-by-month roadmap to help you bridge the skill gaps for Front-end and Full-stack Developer roles:

**Month 1-3: Front-end Fundamentals**

* Month 1:
	+ Learn HTML (structure, syntax, and best practices)
	+ Build small projects, such as a personal website or a simple web page
	+ Resources: W3Schools, Mozilla Developer Network, HTML Dog
* Month 2:
	+ Learn CSS (selectors, properties, and layout)
	+ Build projects that involve styling and layout, such as a simple web application or a responsive website
	+ Resources: W3Schools, CSS-Tricks, Mozilla Developer Network
* Month 3:
	+ Learn JavaScript (basics, DOM manipulation, and events)
	+ Build projects that involve interactive elements, such as a to-do list or a simple game
	+ Resources: Codecademy, FreeCodeCamp, W3Schools

**Month 4-6: Front-end Framework